##### 코랩을 사용할 경우

In [1]:
try:
    # Google Drive를 Colab(/content/drive)rmfja 에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec04
    %ls
except ImportError:
    pass

##### 임포트

In [1]:
import torch
import torch.nn as nn

##### Device 설정

In [2]:
# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cpu


##### 모델 파라미터 확인

In [3]:
class DnnWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        
        # nn.Sequential 사용: 레이어 실행 순서가 명확하게 보임
        self.model = nn.Sequential(
            nn.Linear(10, 64),     # 입력(10) → 은닉층1(64)
            nn.ReLU(),
            nn.Dropout(0.3),       # 30%의 뉴런을 무작위로 비활성화
            
            nn.Linear(64, 32),     # 은닉층1(64) → 은닉층2(32)
            nn.ReLU(),
            nn.Dropout(0.3),       # 30%의 뉴런을 무작위로 비활성화
            
            nn.Linear(32, 2)       # 은닉층2(32) → 출력(2: 자동차/자전거)
        )
    
    def forward(self, x):
        return self.model(x)

# 모델 인스턴스 생성
model = DnnWithDropout().to(device)

# 1. 전체 파라미터 개수
# model.parameters()는 모델의 각 층의 파라미터를 순차적으로 반환하는 제너레이터
# - p.numel(): 해당 층의 파라미터의 총 개수
total_params = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터 개수: {total_params:,}")

# 2. 학습 가능한 파라미터 개수
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 가능한 파라미터 개수: {trainable_params:,}")

# 3. 각 레이어별 파라미터 정보
print("\n=== 레이어별 파라미터 정보 ===")
for name, param in model.named_parameters():
    print(f"{name:20s} | shape: {str(param.shape):20s} | 개수: {param.numel():6,}")

# 4. 모델이 위치한 장치 확인
device = next(model.parameters()).device
print(f"\n모델이 위치한 장치: {device}")

전체 파라미터 개수: 2,850
학습 가능한 파라미터 개수: 2,850

=== 레이어별 파라미터 정보 ===
model.0.weight       | shape: torch.Size([64, 10]) | 개수:    640
model.0.bias         | shape: torch.Size([64])     | 개수:     64
model.3.weight       | shape: torch.Size([32, 64]) | 개수:  2,048
model.3.bias         | shape: torch.Size([32])     | 개수:     32
model.6.weight       | shape: torch.Size([2, 32])  | 개수:     64
model.6.bias         | shape: torch.Size([2])      | 개수:      2

모델이 위치한 장치: cpu
